# Notebook 02 — The Satellite: Where Environmental Data Comes From
**GeoAI for Climate, Environment and Health**

---

> **Where you are in the pipeline**
>
> ```
> 📋 Health records → [🛰 Satellite] → 🌍 Environment → 🔗 Link → ⏱ Timing → 🎯 Results & GeoAI
> ```
>
> You have birth records with locations. Before exploring the climate data,
> it is worth asking: **where does that data come from?**
> This notebook introduces Google Earth Engine, the satellite products we use,
> and the extraction process that produced the pre-built dataset you will use in later notebooks.

---

In [ ]:
%%html
<div id="nb02-prereq-gate" style="font-family:system-ui,sans-serif;margin:0 0 16px"></div>
<script>
(function(){
  var PREV_KEY='nb01_passed';
  var PREV_LABEL='Module 1: Spatial Health Data';
  var gate=document.getElementById('nb02-prereq-gate');
  if(!gate)return;
  if(!localStorage.getItem(PREV_KEY)){
    gate.innerHTML='<div style="background:#fff3cd;border:1px solid #ffc107;border-left:5px solid #ffc107;border-radius:6px;padding:14px 18px;color:#664d03"><b>&#9888; Prerequisite not detected</b><br>It looks like you have not yet passed the quiz for <b>'+PREV_LABEL+'</b>. We recommend completing that module first before continuing here. Your facilitator can help if needed.</div>';
  }else{
    gate.innerHTML='<div style="background:#d1e7dd;border-left:5px solid #198754;border-radius:6px;padding:10px 16px;color:#0a3622"><b>&#10003; '+PREV_LABEL+' completed</b> &#8212; welcome to the next module.</div>';
  }
})();
</script>

---
## Section 1 — Satellite Products for Environmental Health

This notebook introduces the data behind the exposure columns you have been
you will work with from Notebook 03 onwards. Understanding *where* the numbers come from
is essential for honest reporting: every product has a spatial resolution,
a temporal coverage, and systematic gaps.

| Variable | Product | Sensor / Source | Resolution | What it measures |
|---|---|---|---|---|
| `lst_*` | MODIS MOD11A2 | Terra satellite | 1 km / 8-day | Daytime land surface temperature |
| `rain_*` | CHIRPS v2 | Rainfall estimates + gauge merge | ~5 km / monthly | Precipitation |
| `ndvi_*` | MODIS MOD13A3 | Terra satellite | 500 m / monthly | Vegetation greenness (NDVI) |
| `heat_*` | ERA5-Land | ECMWF reanalysis | ~9 km / hourly | 2 m air temperature + humidity index |

**For GIS researchers:** the DHS displaces cluster coordinates by up to 5 km
(urban) or 10 km (rural) to protect participant anonymity. This means a
500 m MODIS pixel and a 5 km displaced cluster may not overlap, but a 25 km
ERA5 grid cell almost certainly still contains the true cluster location.
Coarser resolution products are therefore more robust to the DHS displacement,
a counter-intuitive but important point for study design.

**For public-health researchers:** these are *area-level* exposures assigned
to the DHS cluster centroid, not personal monitors. They capture the ambient
environment, not individual exposure — the ecological fallacy applies, and
within-cluster heterogeneity is invisible to the model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pathlib, sys

def _find_data():
    here = pathlib.Path().resolve()
    for p in (here, here.parent, here.parent.parent):
        if (p / 'data').is_dir():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))  # makes shared_utils importable
            return p / 'data'
    raise FileNotFoundError('Cannot find data/ directory')

DATA_DIR = _find_data()
print(f'Data directory: {DATA_DIR}')

def mc_question(question, options, correct, explanation):
    try:
        import ipywidgets as w
        from IPython.display import display, HTML
        header = w.HTML(f'<div style="font-size:15px;font-weight:600;margin-bottom:10px">🧠 {question}</div>')
        radio = w.RadioButtons(options=[f"{k})  {v}" for k, v in options.items()],
                               layout=w.Layout(width="100%"),
                               style={"description_width": "0px"})
        feedback = w.Output()
        def _check(change):
            if change["name"] != "value": return
            letter = change["new"][0]
            with feedback:
                feedback.clear_output(wait=True)
                if letter == correct:
                    display(HTML(f'<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #28a745">✅ <b>Correct!</b>&nbsp; {explanation}</div>'))
                else:
                    display(HTML(f'<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b>&nbsp; The answer is <b>{correct}</b>. {explanation}</div>'))
        radio.observe(_check, names="value")
        display(w.VBox([header, radio, feedback],
                       layout=w.Layout(border="1px solid #dee2e6", padding="16px 20px",
                                       border_radius="8px", margin="16px 0")))
    except ImportError:
        print(f"Q: {question}")
        for k, v in options.items():
            print(f"  {k}) {v}{' ← CORRECT' if k == correct else ''}")
        print(f"Explanation: {explanation}")


**Conceptual GEE extraction workflow** (no live GEE connection required for this workshop):

```python
import ee
ee.Initialize()

# 1. Upload cluster coordinates as a FeatureCollection (via Assets → New → Table upload)
clusters_fc = ee.FeatureCollection('users/your_username/zw_dhs_clusters')

# 2. Load the MODIS LST image collection, filter to date range
modis_lst = (
    ee.ImageCollection('MODIS/061/MOD11A2')
    .filterDate('2011-01-01', '2015-12-31')
    .select('LST_Day_1km')
    .map(lambda img: img.multiply(0.02).subtract(273.15))  # scale + K→°C
)

# 3. For each image, extract value at each cluster centroid
def sample_image(img):
    return img.reduceRegions(
        collection=clusters_fc,
        reducer=ee.Reducer.mean(),
        scale=1000
    ).map(lambda f: f.set('date', img.date().format('YYYY-MM')))

sampled = modis_lst.map(sample_image).flatten()

# 4. Export to Google Drive
ee.batch.Export.table.toDrive(
    collection=sampled,
    description='zw_modis_lst_2011_2015',
    fileFormat='CSV'
).start()
```

The `zw_climate_monthly.csv` we work with below is the result of exactly this workflow, run for LST (MODIS), rainfall (CHIRPS), NDVI (MODIS), and heat index (derived from ERA5) across all 399 clusters for 60 months.

---
## Section 2 — Live Earth Engine: Try It Yourself

The workflow above is not hypothetical — you can run it right now.
Earth Engine queries are handled by a **server-side proxy** so no personal GEE account,
browser login, or API key is needed. The cell below sets up the query function
and confirms the connection.

> All queries go through the Place Alert Labs API. Only read access to public EE
> datasets is permitted — no data is written to any account.

In [ ]:
import requests, pandas as pd, matplotlib.pyplot as plt

EE_API = 'http://localhost:5000/api/workshop/ee_query'

def ee_query(collection, band, lat, lon, date,
             scale=5000, value_scale=1.0, date_window_days=0):
    """Query EE via the server proxy — no authentication needed."""
    r = requests.post(EE_API, json={
        'collection': collection, 'band': band,
        'lat': lat, 'lon': lon, 'date': date,
        'scale': scale, 'value_scale': value_scale,
        'date_window_days': date_window_days,
    }, timeout=30)
    result = r.json()
    if 'error' in result:
        raise RuntimeError(result['error'])
    return result['value']

# Quick connection test
test = ee_query('UCSB-CHG/CHIRPS/DAILY', 'precipitation',
                lat=-17.303925, lon=31.888711, date='2022-12-15')
print(f'Earth Engine proxy ready ✓   test value: {test} mm')

### Query 1 — What was the rainfall at a DHS cluster on a specific day?

CHIRPS provides daily rainfall estimates at ~5 km resolution.
Below we query a single cluster point for a single day — exactly what the
extraction pipeline does 660,000 times to build the full workshop dataset.

In [ ]:
# Cluster 1: Mashonaland East, rural (-17.30°S, 31.89°E)
lat, lon = -17.303925, 31.888711

rain = ee_query('UCSB-CHG/CHIRPS/DAILY', 'precipitation',
                lat=lat, lon=lon, date='2012-12-25')
print(f'Rainfall at cluster 1 on 2012-11-15: {rain:.1f} mm')

# Try a dry-season date for comparison
rain_dry = ee_query('UCSB-CHG/CHIRPS/DAILY', 'precipitation',
                    lat=lat, lon=lon, date='2012-08-25')
print(f'Rainfall at cluster 1 on 2012-08-15: {rain_dry:.1f} mm  (dry season)')

### Query 2 — Monthly NDVI profile for a cluster

MODIS NDVI shows the seasonal vegetation cycle — green in the wet season,
brown in the dry season. We use the 16-day composite product (MOD13A1)
with a ±16 day window to ensure coverage even during cloud season.

In [ ]:
months = pd.date_range('2013-01-01', '2013-12-01', freq='MS')
ndvi_vals = []
for m in months:
    v = ee_query('MODIS/061/MOD13A1', 'NDVI',
                 lat=lat, lon=lon,
                 date=m.strftime('%Y-%m-%d'),
                 scale=500, value_scale=1e-4, date_window_days=32)
    ndvi_vals.append({'month': m.strftime('%b'), 'ndvi': v or 0})
    print(f'  {m.strftime("%b")}: {v:.3f}' if v else f'  {m.strftime("%b")}: no data')

df_ndvi = pd.DataFrame(ndvi_vals)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(df_ndvi['month'], df_ndvi['ndvi'], color='seagreen', alpha=0.8)
ax.axhline(0.3, color='orange', linestyle='--', label='Sparse vegetation threshold')
ax.set_ylabel('NDVI')
ax.set_title('Cluster 1 — MODIS NDVI monthly, 2013')
ax.legend()
plt.tight_layout(); plt.show()

### Query 3 — ERA5 daily maximum temperature

ERA5-Land provides daily temperature at ~11 km resolution.
October is the pre-monsoon hot season — T3 exposures for November babies
fall almost entirely within this window.

In [ ]:
days = pd.date_range('2011-10-01', '2011-10-31', freq='D')
temp_vals = []
for d in days:
    v = ee_query('ECMWF/ERA5_LAND/DAILY_AGGR', 'temperature_2m_max',
                 lat=lat, lon=lon,
                 date=d.strftime('%Y-%m-%d'), scale=11132)
    temp_c = (v - 273.15) if v else None
    temp_vals.append({'date': d.strftime('%d %b'), 'temp_c': temp_c})

df_t = pd.DataFrame(temp_vals)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(df_t['date'], df_t['temp_c'], marker='o', ms=4, color='tomato')
ax.axhline(35, color='darkred', linestyle='--', label='Heat stress threshold (35°C)')
ax.set_ylabel('Max temperature (°C)')
ax.set_title('Cluster 1 — ERA5 daily max temperature, October 2011')
ax.set_xticklabels(df_t['date'], rotation=45, ha='right', fontsize=8)
ax.legend()
plt.tight_layout(); plt.show()

---
## Section 3 — Load and Validate Pre-extracted Data

**Why pre-extracted?** Running Google Earth Engine live requires a GEE account,
an internet connection, and several minutes per extraction. For a classroom
setting we pre-ran the extraction and stored the result in two files:

- `zw_climate_monthly.csv` — 399 DHS clusters × ~71 months (2010–2016), one row per cluster-month, covering ERA5 temperature, CHIRPS rainfall, MODIS NDVI, and CAMS PM2.5.
- `exposures_zw.duckdb` — the same data in a columnar analytical database
  (DuckDB). DuckDB is like SQLite but built for analytical queries: it reads
  columnar data ~10× faster than CSV for aggregation tasks and supports
  standard SQL. The DuckDB file is the source used in Notebook 06.

We start with the CSV here because it is immediately inspectable. The QC checks
below confirm that the extraction completed without gaps or anomalous values
before we proceed to analysis.

In [ ]:
climate  = pd.read_csv(DATA_DIR / 'zw_climate_monthly.csv')
clusters = pd.read_csv(DATA_DIR / 'zw_clusters.csv')

n_clusters = climate['cluster_id'].nunique()
n_months   = climate.groupby('cluster_id').size().iloc[0]
print(f"Climate data: {len(climate):,} rows  ({n_clusters} clusters × {n_months} months)")
print(f"Year range: {climate['year'].min()}–{climate['year'].max()}")
print(f"Columns: {list(climate.columns)}")
print()

print("Missing values per column:")
print(climate.isnull().sum())
print()

neg_rain = (climate['rain_mm'] < 0).sum()
print(f"Negative rain_mm values: {neg_rain}  (expected: 0)")
if neg_rain > 0:
    print("  WARNING: negative rainfall detected — clip to 0 before analysis.")
else:
    print("  QC passed: all rainfall values are non-negative.")
print()

print("Summary statistics:")
print(climate[['temp_c', 'rain_mm', 'ndvi', 'dust_aod550', 'pm25']].describe().round(2))

---
## Section 4 — Exercise 1 (3 pts): Urban vs Rural Temperature Statistics

The `zw_climate_monthly.csv` already includes the `urban` flag for each cluster. Compute the mean, standard deviation, min, and max of `temp_c` separately for urban and rural clusters.

**Store:**
- `temp_urban_stats` — dict with keys `mean`, `std`, `min`, `max` (floats, 2 dp)
- `temp_rural_stats` — dict with keys `mean`, `std`, `min`, `max` (floats, 2 dp)
- `urban_rural_temp_diff` — float: urban mean − rural mean, 2 dp

In [ ]:
def temp_stats(df, urban_flag):
    sub = df.loc[df['urban'] == urban_flag, 'temp_c']
    return {
        'mean': round(float(sub.mean()), 2),
        'std' : round(float(sub.std()),  2),
        'min' : round(float(sub.min()),  2),
        'max' : round(float(sub.max()),  2),
    }

temp_urban_stats = temp_stats(climate, 1)
temp_rural_stats = temp_stats(climate, 0)
urban_rural_temp_diff = round(temp_urban_stats['mean'] - temp_rural_stats['mean'], 2)

print("Urban clusters:", temp_urban_stats)
print("Rural clusters:", temp_rural_stats)
print(f"Urban – Rural mean temp diff: {urban_rural_temp_diff:+.2f} °C")
print("(Positive = urban warmer; negative = urban cooler due to higher elevation sites)")

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 Google Earth Engine processes satellite imagery on Google's servers rather than downloading it to your computer. What is the primary advantage for African researchers?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q1" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> Results are automatically published</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q1" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> Petabytes of satellite archives can be queried without large local storage, fast internet, or expensive hardware</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q1" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> The analysis is more statistically rigorous</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q1" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> It avoids the need for a Google account</span></label></div>
  <div id="nb02q1-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="Data access is a major barrier in many African research environments. Cloud-based EO platforms shift the computational burden to the provider, making continental-scale analysis possible from a laptop on a slow connection.",ID="nb02q1";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## Section 5 — Worked Example: Temporal Aggregation

We demonstrate two common aggregation patterns:
1. Annual mean per cluster — for cross-sectional exposure assignment.
2. Rolling 3-month mean for a single cluster — showing how trimester windows are conceptually constructed.

In [ ]:
# --- Pattern 1: Annual mean per cluster ---
annual_mean = (
    climate
    .groupby(['cluster_id', 'year'])[['temp_c', 'rain_mm', 'ndvi', 'pm25']]
    .mean()
    .round(3)
    .reset_index()
)
print("Annual mean climate per cluster (first 10 rows):")
print(annual_mean.head(10).to_string(index=False))

**Pattern 2 — Rolling 3-month window (trimester analogue)**

A trimester is approximately 3 consecutive months. The rolling mean below
shows what a trimester-level exposure value *looks like* in the time series
before it is aligned to individual pregnancies. Each point on the orange line
is the average of the current month and the two preceding months — equivalent
to a T3 window ending at that month.

In Notebook 04 these windows were individually aligned to each birth: the
conception date sets `month_0`, and T1/T2/T3 are then calendar-specific
to that pregnancy. The rolling mean here is a population-level illustration,
not the individual-level computation used in the analysis.

**Note for facilitators:** the distinction between population-level seasonal
patterns (this chart) and individual-level exposure windows (Notebook 04) is
a common point of confusion. Both are legitimate analytical choices; they
answer different questions.

In [ ]:
# --- Pattern 2: Rolling 3-month mean for cluster 1 ---
cl1 = climate[climate['cluster_id'] == 1].sort_values(['year', 'month']).copy()
cl1['date_idx'] = range(len(cl1))

cl1['temp_roll3'] = cl1['temp_c'].rolling(3, min_periods=3).mean()
cl1['rain_roll3'] = cl1['rain_mm'].rolling(3, min_periods=3).mean()

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(cl1['date_idx'], cl1['temp_c'],    color='#c0392b', alpha=0.4, lw=1, label='Monthly')
axes[0].plot(cl1['date_idx'], cl1['temp_roll3'], color='#c0392b', lw=2,   label='3-month rolling mean')
axes[0].set_ylabel('Air temp (°C)'); axes[0].legend(fontsize=9)
axes[0].set_title('Cluster 1 — Monthly climate 2011–2015', fontweight='bold')

axes[1].bar(cl1['date_idx'], cl1['rain_mm'],    color='#2980b9', alpha=0.5, width=0.8, label='Monthly')
axes[1].plot(cl1['date_idx'], cl1['rain_roll3'], color='#1a5276', lw=2,   label='3-month rolling mean')
axes[1].set_ylabel('Rainfall (mm/month)'); axes[1].legend(fontsize=9)
axes[1].set_xlabel('Month index (0 = Jan 2011)')

plt.tight_layout()
plt.show()

---
## Section 6 — Exercise 2 (3 pts): Spatial Variability of Rainfall by Calendar Month

For each calendar month (1–12), compute the coefficient of variation (CV = std / mean × 100) of `rain_mm` across all 399 clusters. This captures how spatially heterogeneous rainfall is within Zimbabwe in each month — dry-season months near zero rainfall should show very high CV.

**Store:**
- `rainfall_cv` — pandas Series, index = month (1–12), values = CV (%), rounded to 1 dp
- `most_variable_rainfall_month` — int: the month with the highest CV

In [ ]:
def cv_pct(x):
    m = x.mean()
    return (x.std() / m * 100) if m > 0 else np.nan

rainfall_cv = (
    climate
    .groupby('month')['rain_mm']
    .apply(cv_pct)
    .round(1)
)

most_variable_rainfall_month = int(rainfall_cv.idxmax())

print("Coefficient of variation (%) of rainfall across 399 clusters, by month:")
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
for m, name in enumerate(month_names, 1):
    print(f"  {name}: {rainfall_cv.get(m, float('nan')):.1f}%")
print(f"\nMost spatially variable month: {month_names[most_variable_rainfall_month-1]} (month {most_variable_rainfall_month})")
print("Dry-season months have near-zero mean rainfall, so even tiny amounts produce huge CV.")

**What high spatial CV means:**  when the coefficient of variation across
clusters exceeds 100 % in dry-season months, it tells us that *most* clusters
record near-zero rainfall while a small number receive isolated convective
storms. For exposure assignment this is important: the cluster-level mean
adequately represents a predominantly rainy month, but in dry months a
single storm can dominate the cluster's monthly value while neighbouring
clusters receive nothing. A 5 km CHIRPS pixel may resolve this better than
a 25 km ERA5 pixel, but neither captures sub-pixel variability.

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 Our climate dataset uses monthly averages. What is one important limitation of monthly aggregation for pregnancy exposure research?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q2" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> Monthly data cannot be loaded into Python</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q2" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> Brief extreme events — a 3-day heat spike or flash flood — are diluted and may not appear in the monthly mean</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q2" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> Monthly climate data is less accurate than daily data</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q2" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> DHS surveys only record birth month, so daily data is unnecessary</span></label></div>
  <div id="nb02q2-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="Extreme events (heatwaves, heavy rainfall) drive acute health impacts but are averaged away in monthly summaries. This is a known limitation of ecological exposure studies.",ID="nb02q2";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## Section 7 — Worked Example: Quality Flagging and Gap-Filling

In real satellite and reanalysis data, occasional missing or flagged values arise from sensor issues, orbit gaps, or quality filters. We simulate this by randomly setting 5% of `temp_c` values to NaN, then fill gaps using each cluster's own seasonal mean (month-level mean across all non-missing years).

This approach — seasonal climatological gap-filling — is the standard practice for MODIS and ERA5 data in epidemiological studies.

**Why does cloud cover matter for LST but not CHIRPS?**
MODIS is a passive optical sensor: clouds block the view of the land surface,
producing a fill value rather than an LST measurement. CHIRPS uses microwave
and gauge data that penetrates clouds, so it has no cloud-gap problem.
NDVI also uses MODIS optical bands and has the same issue — cloudy pixels
are flagged and excluded from the monthly composite.

The rainy season (Nov–Mar in Zimbabwe) is when cloud contamination is
highest and LST gaps are most common. This is also when temperatures are
lower — so systematic loss of rainy-season data could bias mean LST upward.
The gap-filling strategy below uses the cluster's own seasonal mean to
impute flagged values, preserving local spatial gradients.

In [ ]:
np.random.seed(99)

climate_sim = climate.copy()

n_missing  = int(len(climate_sim) * 0.05)
miss_idx   = np.random.choice(climate_sim.index, size=n_missing, replace=False)
true_vals  = climate_sim.loc[miss_idx, 'temp_c'].copy()
climate_sim.loc[miss_idx, 'temp_c'] = np.nan

# Fill with cluster × month seasonal mean
seasonal_mean = (
    climate_sim[climate_sim['temp_c'].notna()]
    .groupby(['cluster_id', 'month'])['temp_c']
    .mean()
)
def fill_row(row):
    if pd.isna(row['temp_c']):
        return seasonal_mean.get((row['cluster_id'], row['month']), np.nan)
    return row['temp_c']

climate_sim['temp_c_filled'] = climate_sim.apply(fill_row, axis=1)

rmse = float(np.sqrt(((climate_sim.loc[miss_idx, 'temp_c_filled'] - true_vals) ** 2).mean()))
print(f"Gap-fill RMSE: {rmse:.3f} °C  ({n_missing} values filled from {len(climate_sim):,})")
print("Seasonal climatological gap-fill performs well for temperature (smooth annual cycle).")

---
## Section 8 — Exercise 3 (2 pts): Province-Level PM2.5 — Wet vs Dry Season

Using the `climate` data (which already contains the `province` column), compute the mean `pm25` for each province in:
- **Wet season:** months 11, 12, 1, 2, 3
- **Dry season:** months 5, 6, 7, 8, 9

**Store:**
- `pm25_wet`  — Series: province → mean wet-season PM2.5 (µg/m³), 2 dp
- `pm25_dry`  — Series: province → mean dry-season PM2.5 (µg/m³), 2 dp
- `pm25_diff` — Series: dry − wet, showing the seasonal amplification of air pollution

In [ ]:
WET_MONTHS = [11, 12, 1, 2, 3]
DRY_MONTHS = [5, 6, 7, 8, 9]

wet_df = climate[climate['month'].isin(WET_MONTHS)]
dry_df = climate[climate['month'].isin(DRY_MONTHS)]

pm25_wet  = wet_df.groupby('province')['pm25'].mean().round(2).sort_index()
pm25_dry  = dry_df.groupby('province')['pm25'].mean().round(2).sort_index()
pm25_diff = (pm25_dry - pm25_wet).round(2)

summary = pd.DataFrame({'Wet season (µg/m³)': pm25_wet,
                        'Dry season (µg/m³)': pm25_dry,
                        'Dry − Wet':          pm25_diff})
print(summary.sort_values('Dry − Wet', ascending=False).to_string())
print("\nDry-season PM2.5 is consistently higher — biomass burning and reduced wet deposition.")

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 ERA5 reanalysis has a spatial resolution of ~25 km. DHS clusters are displaced by up to 5 km. What does this mean for our exposure assignments?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q3" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> The 5 km displacement is within one ERA5 grid cell — a displaced cluster is very likely still assigned the same temperature value</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q3" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> ERA5 is too coarse to be useful and should not be used</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q3" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> We must correct all coordinates before analysis</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q3" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> ERA5 resolution automatically matches DHS cluster precision</span></label></div>
  <div id="nb02q3-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="A",E="When the misclassification error (displacement) is smaller than the data resolution (grid cell), it has limited practical impact on the assigned values. This is one reason ERA5 is widely used in DHS-linked exposure studies.",ID="nb02q3";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## Recap

| Concept | Key point |
|---|---|
| Satellite products | MODIS LST → heat; CHIRPS → rainfall; MODIS NDVI → vegetation; ERA5 → heat index |
| GEE workflow | Upload FeatureCollection → join to ImageCollection → `reduceRegions` → export to Drive |
| Urban heat island | Urban LST systematically higher than rural; detectable even at 1 km MODIS resolution |
| Spatial rainfall CV | Highest in dry-season months when most clusters record near-zero but a few receive convective events |
| Quality flagging | Seasonal-mean imputation recovers most cloud-flagged LST values within ±1 °C |
| Wet/dry contrast | Urban industrial areas (e.g., Harare) show the largest seasonal PM2.5 contrast (wet-season wash-out vs dry-season accumulation) |

**Next:** Notebook 03 explores Zimbabwe's climate landscape in detail — seasonal cycles, spatial gradients between Eastern Highlands and lowveld, and the satellite products that capture them.

In [ ]:
%%html
<div id="nb02-scoreboard" style="font-family:system-ui,sans-serif;border:2px solid #adb5bd;border-radius:10px;padding:20px 24px;margin:24px 0;background:#f8f9fa">
  <div style="font-size:17px;font-weight:700;margin-bottom:12px;color:#1a1a2e">&#128202; Module 2 &#8212; Earth Engine &amp; Satellite Data &#8212; Quiz Results</div>
  <div id="nb02-score-display"><p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Complete all reflection questions above to see your score.</p></div>
  <div id="nb02-next-action" style="margin-top:14px"></div>
</div>
<script>
(function(){
  var QUESTIONS=[{id:'nb02q1',correct:'B'},{id:'nb02q2',correct:'B'},{id:'nb02q3',correct:'A'}];
  var PASS=67;
  var NEXT='03_environmental_datasets.ipynb';
  var NEXT_LABEL='Notebook 3: Environmental Datasets';
  var NBID='nb02';
  function update(){
    var correct=0,answered=0;
    QUESTIONS.forEach(function(q){
      var ans=localStorage.getItem('mcq_'+q.id);
      if(ans!==null){answered++;if(ans===q.correct)correct++;}
    });
    var total=QUESTIONS.length;
    var pct=total?Math.round(correct/total*100):0;
    var passed=pct>=PASS;
    var disp=document.getElementById(NBID+'-score-display');
    var act=document.getElementById(NBID+'-next-action');
    if(!disp)return;
    if(answered<total){
      disp.innerHTML='<p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Answer all '+total+' questions above to see your score. ('+(total-answered)+' remaining)</p>';
      if(act)act.innerHTML='';
    }else if(passed){
      disp.innerHTML='<div style="background:#d4edda;color:#155724;padding:12px 16px;border-radius:8px"><b>&#10003; Score: '+correct+'/'+total+' ('+pct+'%) &#8212; MODULE PASSED</b><br><small>Your progress is saved &#8212; you can close and return to this notebook.</small></div>';
      if(act){
        localStorage.setItem(NBID+'_passed','true');
        if(NEXT){act.innerHTML='<a href="'+NEXT+'" style="display:inline-block;background:#198754;color:#fff;padding:10px 22px;border-radius:6px;text-decoration:none;font-weight:600;font-size:14px;margin-top:8px">&#10145; Continue to '+NEXT_LABEL+'</a>';}
        else{act.innerHTML='<div style="background:#cfe2ff;color:#084298;padding:12px 16px;border-radius:8px;margin-top:8px"><b>&#127881; You have completed the full GeoAI pipeline course! Congratulations.</b></div>';}
      }
    }else{
      disp.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:12px 16px;border-radius:8px"><b>&#10007; Score: '+correct+'/'+total+' ('+pct+'%) &#8212; Below pass mark ('+PASS+'%)</b><br><small>Go back to any question and change your answer, then your score updates automatically.</small></div>';
      if(act)act.innerHTML='';
    }
  }
  document.addEventListener('mcq-update',update);
  if(document.readyState==='loading')document.addEventListener('DOMContentLoaded',update);
  else update();
  setInterval(update,3000);
})();
</script>

---

## ✅ Notebook 02 complete

You understand the satellite and reanalysis products behind the climate data — which sensors, which algorithms, what spatial resolution, and what limitations apply. The pre-extracted dataset is ready to explore.

➡️ **Notebook 03** dives into the climate data itself: seasonal patterns, geographic gradients, and the environmental variation that may shape birth outcomes across Zimbabwe.